# 02 — Train the CNN on Kaggle GPU

**Runtime:** Kaggle notebook with **GPU** enabled (Settings → Accelerator → GPU
P100/T4) and the **GTZAN dataset attached** (Add Input → search
`gtzan-dataset-music-genre-classification`).

## What this notebook does
Runs the *exact same repo code* as locally — no duplicated logic — but on
Kaggle's 16GB GPU:
1. clone the project repo,
2. link the attached GTZAN audio to where the config expects it,
3. extract mel-spectrograms (Phase 2 code),
4. train the CNN with AMP on GPU (Phase 4 code),
5. copy `cnn.pt` to `/kaggle/working/` so you can **download it** and commit it
   into the repo / register it in your local MLflow.

The committed split manifest travels with the repo, so Kaggle trains on the
identical train/val/test split as your local classic models.

In [ ]:
# 1. Clone the project repo.
REPO_URL = 'https://github.com/Shashank-ssls/Beat-identifier-MGC.git'
!git clone -q $REPO_URL repo
%cd repo
!python --version && pip -q install librosa==0.10.2.post1 2>/dev/null; echo done

In [ ]:
# 2. Point the config's audio_dir at the attached GTZAN dataset.
#    The Kaggle dataset nests audio under Data/genres_original.
import os, glob, pathlib
cands = glob.glob('/kaggle/input/**/genres_original', recursive=True)
assert cands, 'GTZAN not attached — add it via Add Input'
src = cands[0]
dst = pathlib.Path('data/genres_original')
dst.parent.mkdir(exist_ok=True)
if not dst.exists():
    os.symlink(src, dst)
print('linked', src, '->', dst)
print('genres:', sorted(os.listdir(dst))[:11])

In [ ]:
# 3. Extract mel-spectrograms on Kaggle (the split manifest is already in the repo).
!python -m src.features.extract --what melspec

In [ ]:
# 4. Confirm the GPU is visible, then train. --device auto picks cuda.
import torch; print('cuda?', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
# On 16GB you can safely raise the batch size: --batch-size 32
!python -m src.training.train_cnn --device auto --batch-size 32

In [ ]:
# 5. Stage artifacts for download. Files in /kaggle/working become the
#    notebook's OUTPUT — download cnn.pt and commit it into models/ locally,
#    or save this notebook's output as a Kaggle dataset to attach elsewhere.
import shutil, pathlib
out = pathlib.Path('/kaggle/working'); out.mkdir(exist_ok=True)
for f in ['models/cnn.pt', 'models/cnn_arch.json', 'reports/cm_cnn.png']:
    if pathlib.Path(f).exists():
        shutil.copy(f, out / pathlib.Path(f).name)
print('staged for download:', sorted(p.name for p in out.iterdir()))

## Bringing the model back
1. Download `cnn.pt` (+ `cnn_arch.json`) from the notebook's **Output** tab.
2. Drop them into the repo's `models/` folder locally.
3. Phase 5 (unified eval) and Phase 6 (FastAPI) load `models/cnn.pt` directly —
   no retraining needed locally.